# Data Understanding

## load the dataset

In [2]:
import pandas as pd 
df = pd.read_excel("../data/online_retail.xlsx")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## inspect the dataset

In [9]:
df.info()
df.shape
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386048,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


## check missing values

In [10]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [ ]:
df['CustomerID'].isna().mean()
# 24.9% customerID are missing

0.249266943342886

## data quality check

### not positive quantity

In [14]:
df[df["Quantity"]<=0].head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


In [ ]:
(df["Quantity"]<=0).sum()
# 10624 data entries have nagetive quantity, probably means return or cancel
# invoice beginning with "C" probaly means cancel

10624

In [ ]:
# if all "C" invoice is all negative quantity
df[
    (df["InvoiceNo"].astype(str).str.upper().str.startswith('C'))
    &
    (df['Quantity']>0)
].shape[0]

# YES

0

In [43]:
# if all all non "C" invoice is all positive quantity
df[
    (~df["InvoiceNo"].astype(str).str.upper().str.startswith('C'))
    &
    (df['Quantity']<=0)
].shape[0]
# no, 1336 data entries have non positive quantity with non "C" invoice

1336

In [ ]:
negative_non_c = df[
    (~df['InvoiceNo'].astype(str).str.upper().str.startswith("C"))
    &
    (df['Quantity']<=0)
]

negative_non_c["InvoiceNo"].value_counts().head(20)
# not duplicated data

InvoiceNo
536589    1
562547    1
562544    1
562464    1
562463    1
562391    1
562390    1
562386    1
562385    1
562384    1
562383    1
562382    1
562352    1
562279    1
562278    1
562212    1
562164    1
561927    1
561924    1
562546    1
Name: count, dtype: int64

In [ ]:
negative_non_c[['InvoiceNo','StockCode','Description','Quantity','UnitPrice']].head(20)
# unit price is 0
# assumption: warehouse count correction or wrong data

,InvoiceNo,StockCode,Description,Quantity,UnitPrice
2406,536589,21777,NaN,-10,0.0
4347,536764,84952C,NaN,-38,0.0
7188,536996,22712,NaN,-20,0.0
7189,536997,22028,NaN,-20,0.0
7190,536998,85067,NaN,-6,0.0
7192,537000,21414,NaN,-22,0.0
7193,537001,21653,NaN,-6,0.0
7195,537003,85126,NaN,-2,0.0
7196,537004,21814,NaN,-30,0.0
7197,537005,21692,NaN,-70,0.0


In [ ]:
negative_non_c.describe(include='all')
# this data with null customerID

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
count,1336.0,1336.0,474,1336.000000,1336,1336.0,0.0,1336,1336.0
unique,1336.0,1082.0,138,NaN,NaN,NaN,NaN,1,NaN
top,536589.0,21830.0,check,NaN,NaN,NaN,NaN,United Kingdom,NaN
freq,1.0,5.0,120,NaN,NaN,NaN,NaN,1336,NaN
mean,NaN,NaN,NaN,-154.907934,2011-06-15 11:55:00.314371328,0.0,NaN,NaN,0.0
min,NaN,NaN,NaN,-9600.000000,2010-12-01 16:50:00,0.0,NaN,NaN,-0.0
25%,NaN,NaN,NaN,-84.000000,2011-03-30 16:40:45,0.0,NaN,NaN,-0.0
50%,NaN,NaN,NaN,-30.000000,2011-06-08 11:45:30,0.0,NaN,NaN,-0.0
75%,NaN,NaN,NaN,-8.000000,2011-09-23 14:41:15,0.0,NaN,NaN,0.0
max,NaN,NaN,NaN,-1.000000,2011-12-08 15:24:00,0.0,NaN,NaN,-0.0


In [ ]:
negative_non_c[
    negative_non_c["Description"] == "check"
]
# the most frequently occurring value "check"

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
13217,537425,84968F,check,-20,2010-12-06 15:35:00,0.0,NaN,United Kingdom,-0.0
13218,537426,84968E,check,-35,2010-12-06 15:36:00,0.0,NaN,United Kingdom,-0.0
108577,545546,84249A,check,-150,2011-03-03 15:03:00,0.0,NaN,United Kingdom,-0.0
113580,545990,84598,check,-3000,2011-03-08 13:07:00,0.0,NaN,United Kingdom,-0.0
381676,569875,90195A,check,-45,2011-10-06 15:07:00,0.0,NaN,United Kingdom,-0.0
...,...,...,...,...,...,...,...,...,...
535321,581198,22025,check,-30,2011-12-07 18:26:00,0.0,NaN,United Kingdom,-0.0
535323,581200,84508C,check,-21,2011-12-07 18:27:00,0.0,NaN,United Kingdom,-0.0
535331,581208,72801C,check,-10,2011-12-07 18:35:00,0.0,NaN,United Kingdom,-0.0
535333,581210,23395,check,-26,2011-12-07 18:36:00,0.0,NaN,United Kingdom,-0.0


### not positive price 

In [16]:
df[df["UnitPrice"]<=0].head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom


In [ ]:
(df["UnitPrice"]<=0).sum()
# not positive price with null customerID, not sure why

2517

## revenue
revenue = quantity * unit price

In [18]:
df['Revenue'] = df['Quantity']*df['UnitPrice']

In [ ]:
df['Revenue'].describe()
# abs(min) is equal to max value, probably return order?

count    541909.000000
mean         17.987795
std         378.810824
min     -168469.600000
25%           3.400000
50%           9.750000
75%          17.400000
max      168469.600000
Name: Revenue, dtype: float64

In [20]:
df.nlargest(10,'Revenue')

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,168469.60
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom,77183.60
222680,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,2011-06-10 15:28:00,649.50,15098.0,United Kingdom,38970.00
15017,537632,AMAZONFEE,AMAZON FEE,1,2010-12-07 15:08:00,13541.33,NaN,United Kingdom,13541.33
299982,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom,11062.06
173382,551697,POST,POSTAGE,1,2011-05-03 13:46:00,8142.75,16029.0,United Kingdom,8142.75
348325,567423,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,1412,2011-09-20 11:05:00,5.06,17450.0,United Kingdom,7144.72
52711,540815,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-01-11 12:55:00,2.10,15749.0,United Kingdom,6539.40
160546,550461,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-04-18 13:20:00,2.10,15749.0,United Kingdom,6539.40
421601,573003,23084,RABBIT NIGHT LIGHT,2400,2011-10-27 12:11:00,2.08,14646.0,Netherlands,4992.00


In [21]:
df.nsmallest(10,"Revenue")

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
540422,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom,-168469.60
61624,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2011-01-18 10:17:00,1.04,12346.0,United Kingdom,-77183.60
222681,C556445,M,Manual,-1,2011-06-10 15:31:00,38970.00,15098.0,United Kingdom,-38970.00
524602,C580605,AMAZONFEE,AMAZON FEE,-1,2011-12-05 11:36:00,17836.46,NaN,United Kingdom,-17836.46
43702,C540117,AMAZONFEE,AMAZON FEE,-1,2011-01-05 09:55:00,16888.02,NaN,United Kingdom,-16888.02
43703,C540118,AMAZONFEE,AMAZON FEE,-1,2011-01-05 09:57:00,16453.71,NaN,United Kingdom,-16453.71
15016,C537630,AMAZONFEE,AMAZON FEE,-1,2010-12-07 15:04:00,13541.33,NaN,United Kingdom,-13541.33
16356,C537651,AMAZONFEE,AMAZON FEE,-1,2010-12-07 15:49:00,13541.33,NaN,United Kingdom,-13541.33
16232,C537644,AMAZONFEE,AMAZON FEE,-1,2010-12-07 15:34:00,13474.79,NaN,United Kingdom,-13474.79
524601,C580604,AMAZONFEE,AMAZON FEE,-1,2011-12-05 11:35:00,11586.50,NaN,United Kingdom,-11586.50


## business scale

In [22]:
# number of orders
df["InvoiceNo"].nunique()

25900

In [23]:
# number of customers
df['CustomerID'].nunique()

4372

In [24]:
# number of products
df['StockCode'].nunique()

4070

# Data Cleaning and Feature Engineering

In [76]:
df_clean=df.copy()

In [77]:
df_clean.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
Revenue             0
dtype: int64

## missing customerID

In [78]:
df_clean['CustomerID'].isna().mean()

0.249266943342886

In [79]:
# remove data that missing customerID value
df_clean=df_clean.dropna(subset=["CustomerID"])

In [80]:
df_clean.shape
# 541909-135080=406829

(406829, 9)

## remove cancelled orders (quantity<=0)

In [81]:
df[df["InvoiceNo"].astype(str).str.upper().str.startswith("C")]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom,-27.50
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom,-4.65
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom,-19.80
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom,-6.96
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom,-6.96
...,...,...,...,...,...,...,...,...,...
540449,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397.0,United Kingdom,-9.13
541541,C581499,M,Manual,-1,2011-12-09 10:28:00,224.69,15498.0,United Kingdom,-224.69
541715,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311.0,United Kingdom,-54.75
541716,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09 11:58:00,1.25,17315.0,United Kingdom,-1.25


In [82]:
df[
    (df['InvoiceNo'].astype(str).str.upper().str.startswith("C"))
    &
    (df['Quantity']>0)
].shape[0]

0

In [83]:
df[
    (~df['InvoiceNo'].astype(str).str.upper().str.startswith("C"))
    &
    (df['Quantity']<=0)
].shape[0]

1336

In [84]:
df_clean[
    (~df_clean['InvoiceNo'].astype(str).str.upper().str.startswith("C"))
    &
    (df_clean['Quantity']<=0)
].shape[0]

0

In [85]:
df_clean=df_clean[df_clean['Quantity']>0]

In [86]:
(df_clean['Quantity']<=0).sum()

0

## convert customerID

In [87]:
df_clean['CustomerID'].dtype

dtype('float64')

In [88]:
df_clean['CustomerID'] = (df_clean['CustomerID'].astype(int))

In [89]:
df_clean['CustomerID'].dtype

dtype('int64')

## add date feature

In [90]:
# year
df_clean['Year'] = (df_clean['InvoiceDate'].dt.year)

In [91]:
df_clean['Month'] = (df_clean['InvoiceDate'].dt.month)

In [92]:
df_clean['MonthName'] = (df_clean['InvoiceDate'].dt.month_name)

In [93]:
df_clean['Weekday'] = (df_clean['InvoiceDate'].dt.day_name)

In [94]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 397924 entries, 0 to 541908
Data columns (total 13 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    397924 non-null  object        
 1   StockCode    397924 non-null  object        
 2   Description  397924 non-null  object        
 3   Quantity     397924 non-null  int64         
 4   InvoiceDate  397924 non-null  datetime64[ns]
 5   UnitPrice    397924 non-null  float64       
 6   CustomerID   397924 non-null  int64         
 7   Country      397924 non-null  object        
 8   Revenue      397924 non-null  float64       
 9   Year         397924 non-null  int32         
 10  Month        397924 non-null  int32         
 11  MonthName    397924 non-null  object        
 12  Weekday      397924 non-null  object        
dtypes: datetime64[ns](1), float64(2), int32(2), int64(2), object(6)
memory usage: 39.5+ MB


In [95]:
df_clean.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID,Revenue,Year,Month
count,397924.000000,397924,397924.000000,397924.000000,397924.000000,397924.000000,397924.000000
mean,13.021823,2011-07-10 23:43:36.912475648,3.116174,15294.315171,22.394749,2010.934259,7.612537
min,1.000000,2010-12-01 08:26:00,0.000000,12346.000000,0.000000,2010.000000,1.000000
25%,2.000000,2011-04-07 11:12:00,1.250000,13969.000000,4.680000,2011.000000,5.000000
50%,6.000000,2011-07-31 14:39:00,1.950000,15159.000000,11.800000,2011.000000,8.000000
75%,12.000000,2011-10-20 14:33:00,3.750000,16795.000000,19.800000,2011.000000,11.000000
max,80995.000000,2011-12-09 12:50:00,8142.750000,18287.000000,168469.600000,2011.000000,12.000000
std,180.420210,NaN,22.096788,1713.169877,309.055588,0.247829,3.416527


In [ ]:
# df_clean=drop(df_clean['Revenue_new'])

# Save data

In [ ]:
df_clean.to_csv("../data/cleaned_sales.csv", index=False)

In [98]:
print(df.shape)
print(df_clean.shape)
print(df_clean['Revenue'].sum())
print(df_clean['CustomerID'].nunique())
print(df_clean['InvoiceNo'].nunique())

(541909, 9)
(397924, 13)
8911407.904
4339
18536


# Data Cleaning Summary
The original dataset contained 541,909 data entries. Data validation indentified missing CustomerID, cancelled order and inventory adjustment records that did not represent actual business.

The following cleaning rules were applied:
1. Removed entries with missing CustomerID values.
2. Removed entried with non-positive quantity.

After cleaning, 397,924 valid transaction records remained, representing 18,536 completed orders from 4,339 unique customers.

The cleaned dataset generated a total revenue of £8.91 million and was used for all subsequent sales and customer analyses.